In [1]:
import pandas as pd
import numpy as np

In [2]:
DNDdf = pd.read_csv(r"C:\Users\luisa\Downloads\Global Health Statistics 1\Global Health Statistics.csv")

GDPdf = pd.read_csv(r"C:\Users\luisa\Downloads\API_NY.GDP.MKTP.CD_DS2_en_csv_v2_32\API_NY.GDP.MKTP.CD_DS2_en_csv_v2_32.csv",
    skiprows=4)

In [3]:
GDPdf_drop = GDPdf.drop(columns=[str(year) for year in range(1960, 2000)])

In [6]:
DNDdf['Country'].unique()

array(['Italy', 'France', 'Turkey', 'Indonesia', 'Saudi Arabia', 'USA',
       'Nigeria', 'Australia', 'Canada', 'Mexico', 'China',
       'South Africa', 'Japan', 'UK', 'Russia', 'Brazil', 'Germany',
       'India', 'Argentina', 'South Korea'], dtype=object)

In [8]:
GDPdf_drop['Country Name'].unique()

array(['Aruba', 'Africa Eastern and Southern', 'Afghanistan',
       'Africa Western and Central', 'Angola', 'Albania', 'Andorra',
       'Arab World', 'United Arab Emirates', 'Argentina', 'Armenia',
       'American Samoa', 'Antigua and Barbuda', 'Australia', 'Austria',
       'Azerbaijan', 'Burundi', 'Belgium', 'Benin', 'Burkina Faso',
       'Bangladesh', 'Bulgaria', 'Bahrain', 'Bahamas, The',
       'Bosnia and Herzegovina', 'Belarus', 'Belize', 'Bermuda',
       'Bolivia', 'Brazil', 'Barbados', 'Brunei Darussalam', 'Bhutan',
       'Botswana', 'Central African Republic', 'Canada',
       'Central Europe and the Baltics', 'Switzerland', 'Channel Islands',
       'Chile', 'China', "Cote d'Ivoire", 'Cameroon', 'Congo, Dem. Rep.',
       'Congo, Rep.', 'Colombia', 'Comoros', 'Cabo Verde', 'Costa Rica',
       'Caribbean small states', 'Cuba', 'Curacao', 'Cayman Islands',
       'Cyprus', 'Czechia', 'Germany', 'Djibouti', 'Dominica', 'Denmark',
       'Dominican Republic', 'Algeria',
 

In [13]:
GDPdf_drop.loc[
    GDPdf_drop["Country Name"].str.contains("Tur", case=False, na=False),
    "Country Name"
].unique()

array(['Turks and Caicos Islands', 'Turkmenistan', 'Turkiye'],
      dtype=object)

In [ ]:
country_list = [
    'Italy', 'France', 'Turkiye', 'Indonesia', 'Saudi Arabia',
    'United States', 'Nigeria', 'Australia', 'Canada', 'Mexico',
    'China', 'South Africa', 'Japan', 'United Kingdom',
    'Russian Federation', 'Brazil', 'Germany', 'India',
    'Argentina', 'Korea, Rep.'
]

In [15]:
GDPdf_filtered = GDPdf_drop[
    GDPdf_drop["Country Name"].isin(country_list)
]

In [ ]:
GDPdf_filtered["Country Name"].unique()


array(['Argentina', 'Australia', 'Brazil', 'Canada', 'China', 'Germany',
       'France', 'United Kingdom', 'Indonesia', 'India', 'Italy', 'Japan',
       'Korea, Rep.', 'Mexico', 'Nigeria', 'Russian Federation',
       'Saudi Arabia', 'Turkiye', 'United States', 'South Africa'],
      dtype=object)

In [ ]:
# 1) GDP wide -> long 
GDP_long = GDPdf_filtered.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
    value_name="GDP"
)

GDP_long["Year"] = pd.to_numeric(GDP_long["Year"], errors="coerce")
GDP_long = GDP_long.dropna(subset=["Year"])
GDP_long["Year"] = GDP_long["Year"].astype(int)

GDP_long = GDP_long.rename(columns={"Country Name": "Country"})
GDP_long = GDP_long[GDP_long["Indicator Code"] == "NY.GDP.MKTP.CD"]

# keep only the years DNDdf has
GDP_long = GDP_long[GDP_long["Year"].between(2000, 2024)]

# 2) make DNDdf country names match World Bank 
DNDdf_fixed = DNDdf.copy()
DNDdf_fixed["Country"] = DNDdf_fixed["Country"].replace({
    "USA": "United States",
    "UK": "United Kingdom",
    "Russia": "Russian Federation",
    "South Korea": "Korea, Rep.",
    "Turkey": "Turkiye"
})

# 3) merge GDP into DNDdf
DNDdf_with_gdp = DNDdf_fixed.merge(
    GDP_long[["Country", "Year", "GDP"]],
    on=["Country", "Year"],
    how="left"
)

DNDdf_with_gdp.head()

,Country,Year,Disease Name,Disease Category,Prevalence Rate (%),Incidence Rate (%),Mortality Rate (%),Age Group,Gender,Population Affected,...,Treatment Type,Average Treatment Cost (USD),Availability of Vaccines/Treatment,Recovery Rate (%),DALYs,Improvement in 5 Years (%),Per Capita Income (USD),Education Index,Urbanization Rate (%),GDP
0,Italy,2013,Malaria,Respiratory,0.95,1.55,8.42,0-18,Male,471007,...,Medication,21064,No,91.82,4493,2.16,16886,0.79,86.02,2.153226e+12
1,France,2002,Ebola,Parasitic,12.46,8.63,8.75,61+,Male,634318,...,Surgery,47851,Yes,76.65,2366,4.82,80639,0.74,45.52,1.492428e+12
2,Turkiye,2015,COVID-19,Genetic,0.91,2.35,6.22,36-60,Male,154878,...,Vaccination,27834,Yes,98.55,41,5.81,12245,0.41,40.20,8.654601e+11
3,Indonesia,2011,Parkinson's Disease,Autoimmune,4.68,6.29,3.99,0-18,Other,446224,...,Surgery,144,Yes,67.35,3201,2.22,49336,0.49,58.47,8.929691e+11
4,Italy,2013,Tuberculosis,Genetic,0.83,13.59,7.01,61+,Male,472908,...,Medication,8908,Yes,50.06,2832,6.93,47701,0.50,48.14,2.153226e+12


In [27]:
DNDdf_with_gdp

,Country,Year,Disease Name,Disease Category,Prevalence Rate (%),Incidence Rate (%),Mortality Rate (%),Age Group,Gender,Population Affected,...,Treatment Type,Average Treatment Cost (USD),Availability of Vaccines/Treatment,Recovery Rate (%),DALYs,Improvement in 5 Years (%),Per Capita Income (USD),Education Index,Urbanization Rate (%),GDP
0,Italy,2013,Malaria,Respiratory,0.95,1.55,8.42,0-18,Male,471007,...,Medication,21064,No,91.82,4493,2.16,16886,0.79,86.02,2.153226e+12
1,France,2002,Ebola,Parasitic,12.46,8.63,8.75,61+,Male,634318,...,Surgery,47851,Yes,76.65,2366,4.82,80639,0.74,45.52,1.492428e+12
2,Turkiye,2015,COVID-19,Genetic,0.91,2.35,6.22,36-60,Male,154878,...,Vaccination,27834,Yes,98.55,41,5.81,12245,0.41,40.20,8.654601e+11
3,Indonesia,2011,Parkinson's Disease,Autoimmune,4.68,6.29,3.99,0-18,Other,446224,...,Surgery,144,Yes,67.35,3201,2.22,49336,0.49,58.47,8.929691e+11
4,Italy,2013,Tuberculosis,Genetic,0.83,13.59,7.01,61+,Male,472908,...,Medication,8908,Yes,50.06,2832,6.93,47701,0.50,48.14,2.153226e+12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,Saudi Arabia,2021,Parkinson's Disease,Infectious,4.56,4.83,9.65,0-18,Female,119332,...,Vaccination,4528,Yes,92.11,1024,3.88,29335,0.75,27.94,9.826611e+11
999996,Saudi Arabia,2013,Malaria,Respiratory,0.26,1.76,0.56,0-18,Female,354927,...,Surgery,20686,No,84.47,202,7.95,30752,0.47,77.66,7.697557e+11
999997,United States,2016,Zika,Respiratory,13.44,14.13,1.91,19-35,Other,807915,...,Therapy,18807,No,86.81,3338,7.31,62897,0.72,46.90,1.869511e+13
999998,Nigeria,2020,Asthma,Chronic,1.96,14.56,4.98,61+,Female,385896,...,Medication,21033,Yes,62.15,4806,3.82,98189,0.51,34.73,5.985868e+11


In [30]:
missing_countries = (
    DNDdf_with_gdp.loc[DNDdf_with_gdp["GDP"].isna(), "Country"]
    .value_counts()
    .head(30)
)
missing_countries

Series([], Name: count, dtype: int64)

In [31]:
DNDdf_with_gdp.isnull().sum()

Country                               0
Year                                  0
Disease Name                          0
Disease Category                      0
Prevalence Rate (%)                   0
Incidence Rate (%)                    0
Mortality Rate (%)                    0
Age Group                             0
Gender                                0
Population Affected                   0
Healthcare Access (%)                 0
Doctors per 1000                      0
Hospital Beds per 1000                0
Treatment Type                        0
Average Treatment Cost (USD)          0
Availability of Vaccines/Treatment    0
Recovery Rate (%)                     0
DALYs                                 0
Improvement in 5 Years (%)            0
Per Capita Income (USD)               0
Education Index                       0
Urbanization Rate (%)                 0
GDP                                   0
dtype: int64

In [29]:
DNDdf_with_gdp.to_csv(
    r"C:\Users\luisa\Downloads\DNDdf_with_GDP.csv",
    index=False
)